# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_json = dataset.metadata.to_json()
print(f"{metadata_json['name']}: {metadata_json['description']}")

## 2. Data Overview
Review available record sets, their fields, and the corresponding `@id`s. Each record set, field, and column is referenced by its `@id` to ensure unambiguous referencing in Croissant.

In [ ]:
# Show all available record sets by their @id
print('Available record sets:')
record_sets = [rs['@id'] for rs in dataset.metadata['recordSet']]
for rs in dataset.metadata['recordSet']:
    print(f"RecordSet: {rs['@id']} | name: {rs.get('name','#N/A')}")
    print('  Fields:')
    for field in rs.get('field', []):
        if isinstance(field, dict):
            print(f"    {field['@id']}  | name: {field.get('name','#N/A')}")
        else:
            print(f"    {field} (See schema for definition)")
    print('  Columns:')
    for column in rs.get('column', []):
        if isinstance(column, dict):
            print(f"    {column['@id']}  | name: {column.get('name','#N/A')}")
        else:
            print(f"    {column} (See schema for definition)")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# List all record set @id values
all_record_set_ids = [rs['@id'] for rs in dataset.metadata['recordSet']]
dataframes = {}

for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))  # Each record is a dict with field @id keys
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"RecordSet '{record_set_id}' loaded ({df.shape[0]} records, columns: {list(df.columns)})")
    else:
        print(f"RecordSet '{record_set_id}' loaded (0 records)")

# For demonstration, pick the first non-empty record set
main_record_set_id = next(iter(dataframes)) if dataframes else None
if main_record_set_id:
    print(f"\nAvailable columns in {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print('No non-empty record sets found.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Note: Use the entity `@id` for all fields as per Croissant best practices.

For demonstration, we select the first numeric field found in the first loaded DataFrame.

In [ ]:
import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id].copy()
    # Try to find a numeric column: Check datatype, fallback to first column
    numeric_field_id = None
    for col in df.columns:
        # Try to convert to numeric to determine numerics
        try:
            sample = pd.to_numeric(df[col], errors='coerce')
            if sample.notnull().sum() > 0:
                numeric_field_id = col
                break
        except Exception:
            pass
    if not numeric_field_id:
        print('No suitable numeric field found.')
    else:
        # Clean/convert
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0  # Use mean as threshold if possible
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by the next available column
        group_field = None
        colnames = list(df.columns)
        numeric_idx = colnames.index(numeric_field_id)
        if len(colnames) > numeric_idx + 1:
            group_field = colnames[numeric_idx + 1]
        if group_field and group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print('No suitable group field found.')
else:
    print('No DataFrame loaded to perform EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we show a histogram of the selected numeric field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
else:
    print('No numeric field to visualize.')

## 6. Conclusion
In this notebook, we demonstrated how to load, examine, and process a Croissant-compliant dataset using the `mlcroissant` library.

- All record sets, fields, and columns were referenced by their `@id`, as per best practice.
- We loaded all available record sets and performed a basic exploration, filtering, normalization, and grouping on one numeric field.
- A simple distribution visualization was shown using matplotlib.

Refer to the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset and Croissant documentation for further details and advanced workflows for scientific and FAIR dataset analysis.